# 005 — SQL Runner with CodeMirror · 데모

Jupyter 노트북에서 동작하는 single-file SQL 편집기 + 실행 위젯.

| 단계 | 무엇을 보나 |
|---|---|
| 1️⃣ 환경 준비 | 임시 SQLite DB + 4개 테이블 |
| 2️⃣ `runner.show()` | CodeMirror 에디터 + 자동완성 + ▶ 실행 UI |
| 3️⃣ `runner.history()` | 날짜 sidebar + SQL/전체 [📋 복사] · 새 세션에서도 자동으로 이어짐 |
| 4️⃣ `on_execute` 콜백 직접 등록 | sqlite/list/dict/REST 등 임의 백엔드 |
| 5️⃣ 대량 스키마 stress | 사내 금융사급 113 테이블 / 1,800+ 컬럼 |


---

## 1️⃣ 환경 준비 — DB + runner

`with_sqlite` 헬퍼로 한 줄에 셋업. 각 컬럼에는 `doc` 으로 한국어 설명을 덧붙여 추천 popup / 트리 hover 에 노출되게 합니다.


In [ ]:
import os, sys, sqlite3, tempfile
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))
from sql_codemirror import SQLRunnerCM

# ── 임시 DB + 데모 데이터 ──
db_path = os.path.join(tempfile.gettempdir(), 'sql_codemirror_demo.db')
if os.path.exists(db_path):
    os.remove(db_path)
with sqlite3.connect(db_path) as conn:
    conn.executescript("""
    CREATE TABLE users (
        id INTEGER PRIMARY KEY, name TEXT, email TEXT,
        region TEXT, plan_type TEXT, signup_at TIMESTAMP
    );
    CREATE TABLE orders (
        id INTEGER PRIMARY KEY, user_id INTEGER REFERENCES users(id),
        amount REAL, status TEXT, ordered_at TIMESTAMP
    );
    CREATE TABLE products (
        id INTEGER PRIMARY KEY, sku TEXT UNIQUE,
        name TEXT, category TEXT, price REAL, stock INTEGER
    );
    CREATE TABLE events (
        id INTEGER PRIMARY KEY, user_id INTEGER,
        event_type TEXT, value REAL, occurred_at TIMESTAMP
    );
    INSERT INTO users VALUES
      (1,'김알리스','alice@ex.com','서울','pro','2026-01-10'),
      (2,'이밥','bob@ex.com','부산','free','2026-02-15'),
      (3,'박찰리','charlie@ex.com','서울','enterprise','2026-03-01'),
      (4,'최다나','dana@ex.com','대전','pro','2026-03-12'),
      (5,'정에반','evan@ex.com','서울','free','2026-04-02');
    INSERT INTO orders VALUES
      (1,1,15000,'paid','2026-04-01'),(2,1,8500,'paid','2026-04-05'),
      (3,1,93000,'paid','2026-04-12'),(4,2,29000,'paid','2026-04-08'),
      (5,3,15000,'cancelled','2026-04-10'),(6,4,117000,'paid','2026-04-15');
    INSERT INTO products VALUES
      (1,'SKU-001','노트북','electronics',1850000,12),
      (2,'SKU-002','마우스','electronics',45000,128);
    """)

runner = SQLRunnerCM.with_sqlite(db_path)
# 컬럼별 한국어 description (자동 추출된 type 위에 보강)
runner.add_table('users', [
    {'name':'id','type':'INTEGER','doc':'PK'},
    {'name':'name','type':'TEXT','doc':'표시 이름'},
    {'name':'email','type':'TEXT','doc':'로그인용 이메일'},
    {'name':'region','type':'TEXT','doc':'거주 지역'},
    {'name':'plan_type','type':'TEXT','doc':'free / pro / enterprise'},
    {'name':'signup_at','type':'TIMESTAMP','doc':'가입 시각'},
], description='사용자 마스터')
runner.add_table('orders', [
    {'name':'id','type':'INTEGER','doc':'PK'},
    {'name':'user_id','type':'INTEGER','doc':'FK → users.id'},
    {'name':'amount','type':'REAL','doc':'결제 금액 KRW'},
    {'name':'status','type':'TEXT','doc':'paid · cancelled · pending'},
    {'name':'ordered_at','type':'TIMESTAMP','doc':'주문 시각'},
], description='주문 내역')

print(f'✓ DB · runner 준비 완료 ({len(runner.tables)} 테이블)')


---

## 2️⃣ `runner.show()` — 메인 UI

CodeMirror 에디터 + 좌측 entity 트리 + 컨텍스트 자동완성 + ▶ 실행 버튼이 한 셀에 통합됩니다.

**시도해보세요**:
- `Cmd/Ctrl + Enter` → ▶ 실행
- `SELECT * FROM ` 입력 → popup 자동완성에 4 테이블 후보
- `SELECT u.id, u.name FROM users u WHERE u.` → users 컬럼만 매치
- 좌측 entity 트리 → 테이블/컬럼 클릭 시 커서 위치에 인서트
- 잘못된 SQL (예: 괄호 미닫힘) → 빨간 ❌ 메시지 + ▶ 실행 버튼 disabled


In [ ]:
runner.set_query(
    "-- ▶ 실행 (Cmd/Ctrl+Enter) · Ctrl+Space 자동완성\n"
    "SELECT u.name, u.region, COUNT(o.id) AS n_orders, SUM(o.amount) AS total\n"
    "FROM users u LEFT JOIN orders o ON o.user_id = u.id\n"
    "WHERE o.status = 'paid'\n"
    "GROUP BY u.id ORDER BY total DESC NULLS LAST;"
)
runner.show()


---

## 3️⃣ `runner.history()` — 실행 이력 + 날짜 sidebar

▶ 실행을 누를 때마다 자동 누적. `runner.history()` 호출 시:

- 📅 좌측 날짜 sidebar (클릭 → 해당 날짜 entry 만)
- 📋 항목별 / 전체 SQL 복사 버튼
- 🔁 새 노트북 세션에서도 자동으로 이어짐


In [ ]:
# 데모용 — 5번 실행해 history 채우기 (실제로는 ▶ 실행 클릭이 자동 누적)
import time
demo_queries = [
    "SELECT COUNT(*) AS total_users FROM users",
    "SELECT region, COUNT(*) AS n FROM users GROUP BY region",
    ("SELECT u.name, SUM(o.amount) AS total\n"
     "  FROM users u JOIN orders o ON o.user_id = u.id\n"
     "  WHERE o.status = 'paid'\n"
     "  GROUP BY u.id ORDER BY total DESC LIMIT 5"),
    "SELECT * FROM products WHERE price > 50000",
    "SELECT * FROM nonexistent_table",   # 의도적 에러
]
for q in demo_queries:
    try:
        runner.execute(q)
    except Exception:
        pass   # 에러도 history 에 자동 누적
    time.sleep(0.4)

print(f"누적: {len(runner.history)} 건")


In [ ]:
runner.history()


---

## 4️⃣ `on_execute` 콜백 직접 등록 — 임의 백엔드

`SQLRunnerCM.with_sqlite()` 는 SQLite + pandas 한정 헬퍼. 사내 SQL 엔진 / mock dict / REST API / DataFrame 등 **임의 함수** 를 `on_execute=fn` 으로 주입하면 같은 위젯이 그 백엔드 위에서 동작합니다. 시그니처: `(sql: str) -> Any`.


In [ ]:
# 사례 A — sqlite raw cursor (pandas 미의존, 결과는 list[dict])
def run_raw(sql: str):
    with sqlite3.connect(db_path) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(r) for r in conn.execute(sql).fetchall()]

# 사례 B — 메모리 mock 엔진 (DB 없이 dict 위에서)
MEMORY = {
    'fruit': [{'id': 1, 'name': '사과', 'price': 3000},
              {'id': 2, 'name': '바나나', 'price': 1500}],
}
def run_memory(sql: str):
    # 실 사내 엔진은 SQL 파싱 → 적절한 dict 조회. 데모는 단순 로깅 + 첫 키 반환.
    print(f'[MOCK] {sql.strip()[:60]}')
    return MEMORY['fruit']

# 사례 C — REST/사내 엔진 wrapper (실 코드는 requests / 사내 SDK 사용)
def run_via_api(sql: str):
    # response = requests.post('http://internal.api/sql', json={'sql': sql})
    # return response.json()['rows']
    return [{'engine': 'internal', 'sql_received': sql, 'rows_returned': 0}]

# 셋 중 하나 골라 띄워보기 — 트리 스키마는 수동 등록
runner_b = SQLRunnerCM(on_execute=run_raw, history_dir=None)
runner_b.from_dict({
    'users':    ['id', 'name', 'email', 'region'],
    'orders':   ['id', 'user_id', 'amount', 'status'],
    'products': ['id', 'sku', 'name', 'price'],
})
runner_b.set_query('SELECT name, region FROM users LIMIT 5;')
runner_b.show()


---

## 5️⃣ 대량 스키마 stress — 113 테이블 (사내 금융사 시뮬)

좌측 entity 패널은 위젯 객체 대신 단일 HTML 한 덩어리로 렌더하므로 수천 개 entity 도 첫 렌더 지연이 거의 없습니다. **검색창** 에 입력하면 JS 자체 필터로 즉석 매치 (Python round-trip 0).

**검색 시도**: `customer` (9개 테이블) · `amount` (37개 컬럼) · `kyc` · `portfolio_`


In [ ]:
import random
random.seed(42)

def fake_engine(sql: str):
    return [{"row_id": i, "value": f"val_{i:03d}", "score": i * 1.5}
            for i in range(20)]

SCHEMA_GROUPS = {
    "고객": ["customers","customer_addresses","customer_contacts",
            "customer_documents","customer_consents","customer_segments",
            "customer_relationships","customer_notes","kyc_records",
            "beneficial_owners","customer_risk_profiles"],
    "계좌": ["accounts","account_holders","account_balances",
            "account_statements","account_transactions","account_holds",
            "account_overdrafts","account_fees","account_limits",
            "account_types","account_alerts","savings_accounts",
            "checking_accounts","credit_accounts","investment_accounts"],
    "거래": ["transactions","trade_executions","settlements","clearings",
            "wire_transfers","internal_transfers","fee_transactions",
            "currency_exchanges","dividend_payments","interest_payments",
            "transaction_journal","pending_transactions","reversals"],
    "주문": ["orders","order_executions","order_routing","order_book",
            "order_states","order_amendments","order_cancellations",
            "basket_orders","algorithmic_orders","dark_pool_orders",
            "smart_routing_logs"],
    "상품/증권": ["instruments","equity_securities","fixed_income_securities",
                "derivatives","options","futures","swaps","mutual_funds",
                "etfs","structured_products","instrument_master",
                "instrument_prices","instrument_dividends","corporate_actions"],
    "포트폴리오": ["portfolios","portfolio_holdings","portfolio_allocations",
                "portfolio_returns","portfolio_metrics","portfolio_constraints",
                "portfolio_rebalances","portfolio_risk","portfolio_benchmarks"],
    "컴플라이언스": ["aml_screenings","sanctions_lists","watchlists",
                "suspicious_activities","compliance_alerts",
                "regulatory_filings","audit_trails","ofac_screenings",
                "pep_records","compliance_cases"],
    "분석/이벤트": ["events","page_views","sessions","user_journeys",
                "funnel_steps","conversion_events","click_streams",
                "error_events","performance_metrics","ab_test_assignments"],
    "참조데이터": ["currencies","exchange_rates","countries","business_calendars",
                "holidays","market_data_sources","reference_indices",
                "sector_classifications","gics_codes","isin_database"],
    "시스템": ["system_users","system_roles","system_permissions",
            "system_sessions","api_keys","audit_logs","system_logs",
            "error_logs","access_logs","system_configuration"],
}

NUMERIC = ["amount","balance","fee","rate","score","quantity","volume",
           "weight","factor","threshold","premium","discount","tax",
           "interest_amount","principal","yield_rate","spread","ratio",
           "percentage","market_value","book_value","nav","price","unit_cost"]
TEXT = ["name","code","description","category","type","status","label",
        "tag","notes","comment","reason","source","channel","method",
        "currency_code","country_code","language_code","external_id",
        "reference_no","approval_code"]
DATE = ["created_at","updated_at","deleted_at","effective_date",
        "expiration_date","settled_at","executed_at","approved_at",
        "submitted_at","processed_at","cancelled_at","valued_at"]
INT = ["version","sequence_no","priority","max_retries","attempt_count",
       "row_count","line_no"]
FK = ["customer","account","order","transaction","instrument","portfolio",
      "product","user"]

def gen_cols(tname, count):
    cols = [{"name":"id","type":"INTEGER","doc":"PK"}]
    used = {"id"}
    for prefix in FK:
        if prefix in tname.split("_") and not tname.startswith(prefix):
            fk = f"{prefix}_id"
            if fk not in used:
                cols.append({"name":fk,"type":"INTEGER","doc":f"FK → {prefix}s.id"})
                used.add(fk)
    pools = [(NUMERIC,"REAL"),(TEXT,"TEXT"),(DATE,"TIMESTAMP"),(INT,"INTEGER")]
    while len(cols) < count:
        pool, ctype = random.choice(pools)
        cand = random.choice(pool)
        if cand not in used:
            used.add(cand)
            cols.append({"name":cand,"type":ctype,"doc":f"{cand} ({ctype.lower()})"})
    return cols

big = SQLRunnerCM(on_execute=fake_engine, history_dir=None)
total_t = total_c = 0
for group, tables in SCHEMA_GROUPS.items():
    for t in tables:
        n = random.randint(8, 25)
        big.add_table(t, gen_cols(t, n), description=f"[{group}] {t.replace('_',' ')}")
        total_t += 1; total_c += n

print(f"✓ {len(SCHEMA_GROUPS)} 도메인 · {total_t} 테이블 · {total_c} 컬럼 = {total_t+total_c} entities")
big.set_query("-- 검색창에 customer / amount / kyc / portfolio_ 입력해보세요\n"
              "SELECT * FROM customers LIMIT 20;")
big.show()


---

## 정리

| 기능 | 사용법 |
|---|---|
| ▶ 실행 + 결과 표 렌더 | `runner.show()` → Cmd/Ctrl+Enter |
| 다음 셀 분석 | `runner.last_result` · `runner.last_query` · `runner.history[-1]` |
| 이력 시각화 + 복사 | `runner.history()` · `runner.history(n=10)` · `runner.history.to_markdown()` |
| 외부 호출 | `runner.execute(sql)` (내부에서 history 누적까지 자동) |
| 임의 백엔드 | `SQLRunnerCM(on_execute=fn)` — sqlite/list/dict/REST 모두 가능 |
| 사내 정책 검증 | `SQLRunnerCM(on_validate=fn)` — 잘못된 SQL 시 ▶ 실행 disabled |

## 폐쇄망 친화 체크

| 항목 | 상태 |
|---|---|
| 외부 네트워크 / CDN | ❌ 없음 (모든 CSS/JS 인라인) |
| `<link href>` / `<script src>` | ❌ 없음 |
| 새 서버 / 포트 | ❌ 없음 |
| 바이너리 영속화 | ❌ 없음 (history 는 텍스트 JSONL) |
| 단일 반입 단위 | `sql_codemirror.py` 한 파일 |
